# Assignment 3 — English→Hindi MT: results walkthrough

This notebook only **reads** artifacts produced by the `scripts/` pipeline; it does not retrain anything.
Run `scripts/run_all.ps1` first (or use the committed results).

Sections: dataset stats → headline metrics → model comparison → temperature → error examples.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RES = ROOT / 'results'
print('project root:', ROOT)

## 1. Dataset and split audit

IIT Bombay English–Hindi corpus, official provider splits, fixed 60k-pair shared training subset.

In [ ]:
stats = json.loads((RES / 'data_stats.json').read_text(encoding='utf-8'))
display(pd.DataFrame({k: {'pairs': v['pairs'], 'src words (mean)': v['src_words']['mean'], 'tgt words (mean)': v['tgt_words']['mean']} for k, v in stats['splits'].items()}).T)
stats['leakage_audit']

## 2. Headline results

BLEU (13a) and chrF with 95% sentence-level bootstrap CIs on the same 2,507-sentence test set.

In [ ]:
df = pd.read_csv(RES / 'summary_table.csv')
display(df[['system','architecture','params_m','train_or_adapt_s','inference_s','bleu','chrf','ter']])
display(Image(filename=str(RES / 'figures' / 'bleu_by_system.png')))
display(Image(filename=str(RES / 'figures' / 'bleu_vs_params.png')))

## 3. Generated translations (inspect, do not trust aggregate scores alone)

Every system translated the identical test rows in the identical order.

In [ ]:
import itertools
def load(name):
    path = RES / 'translations' / f'{name}.jsonl'
    return [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines()] if path.exists() else []

systems = ['basic_transformer','mt5_base','llama_prompt3shot','llama_qlora','gemma_prompt3shot','gemma_qlora']
rows = {s: load(s) for s in systems}
for i in range(3):
    print('SRC :', rows['basic_transformer'][i]['src'])
    print('REF :', rows['basic_transformer'][i]['ref'])
    for s in systems:
        if rows[s]:
            print(f'{s:18s}:', rows[s][i]['hyp'])
    print('-' * 100)

## 4. Sampling-temperature analysis

Greedy decoding usually maximises adequacy; higher temperatures add diversity but also drift and hallucination.

In [ ]:
temp_path = RES / 'temperature' / 'temperature_results.json'
if temp_path.exists():
    temp = json.loads(temp_path.read_text(encoding='utf-8'))
    for system, rows_ in temp.items():
        print(system)
        display(pd.DataFrame(rows_)[['temperature','bleu','chrf','distinct_1','distinct_2','mean_words','empty_rate']])
    display(Image(filename=str(RES / 'figures' / 'temperature_analysis.png')))

## 5. Responsible-AI screening

Automatic flags are screening aids; the qualitative judgements in `docs/report.md` come from reading the outputs in `results/error_analysis/flagged_examples.md`.

In [ ]:
flags = json.loads((RES / 'error_analysis' / 'risk_flags.json').read_text(encoding='utf-8'))
display(pd.DataFrame(flags).T[['n','no_devanagari','dropped_number_sentences','added_number_sentences','untranslated_latin_sentences','repetition_loops']])